# NeuroLens — auditable progression modelling on real open data

**NeuroLens is a research prototype.** It is not clinically validated, must not be used for
diagnosis or treatment decisions, and its feature attributions describe *model association*,
not biological causation.

This notebook runs the complete audit pipeline — ingestion, leakage-free splits, two prediction
tasks, calibration and conformal intervals, OOD detection, a fairness audit with adequacy
statuses, mitigation comparison, explanations, and the audit report — end to end on **real**
data.

### Data policy

NeuroLens never generates data. There is no synthetic cohort anywhere in the project, including
as a fallback. If the real dataset cannot be reached, cells below fail loudly rather than
quietly inventing rows. See `DATA.md` for the full provenance of every dataset used here and
for the access barrier that separates the open pathway from the Huntington's disease pathway.


## 1. Environment

NeuroLens is pure Python with NumPy/pandas/scikit-learn; no build step and no GPU requirement.
A GPU runtime helps only if you swap in a heavier model family.


In [ ]:
# If running from a clone, install in place; otherwise pip-install from the repository.
import os, subprocess, sys

if os.path.isdir('neurolens'):
    os.chdir('neurolens')

if not os.path.isdir('src/neurolens'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'git+https://github.com/snehapadgaonkar/neurolens.git'], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

sys.path.insert(0, os.path.abspath('src'))
import neurolens
print('neurolens', neurolens.__version__)


## 2. Fetch the real dataset

Dallas Lifespan Brain Study (OpenNeuro `ds004856`), **CC0 public domain**, no account and no data
use agreement. This pulls the participants table and its data dictionary — 464 real participants
with three waves of MMSE.

The DLBS imaging and genetic derivatives live in OpenNeuro's S3 bucket and need the AWS CLI:

```bash
aws s3 sync --no-sign-request s3://openneuro.org/ds004856 data/raw/ds004856
```

Without them the run proceeds on clinical + demographic features and **records the missing
modalities** in the QC ledger and the limitations section. It never imputes them.


In [ ]:
from neurolens.datasets import fetch_openneuro_tabular, MANIFESTS, resolve_manifest

for key, manifest in MANIFESTS.items():
    print(f"{key:<24} {manifest.access:<14} redistributable={manifest.redistributable}")

print()
written = fetch_openneuro_tabular('ds004856', 'data/raw/ds004856')
present = {name: os.path.getsize(os.path.join('data/raw/ds004856', name))
           for name in ('participants.tsv', 'participants.json',
                        'dataset_description.json')
           if os.path.exists(os.path.join('data/raw/ds004856', name))}
assert present, 'the real participants table could not be obtained'
print('downloaded this call:', {k: os.path.getsize(v) for k, v in written.items()})
print('available on disk (bytes):', present)


## 3. Load the cohort and check what arrived

The loader preserves the dataset's own codes; the cohort builder applies the published data
dictionary. Where the published code list does not match the data — as with DLBS `Ethnicity` —
NeuroLens logs the mismatch and keeps the raw codes instead of guessing a mapping.


In [ ]:
import pandas as pd
from neurolens.config import NeuroLensConfig
from neurolens.datasets import load_cohort

config = NeuroLensConfig()
tables = load_cohort(config.data.dataset, config.data.data_dir)
print(tables.summary())
print()
print('QC reasons:', tables.qc.summary())
tables.patients.head()


## 4. Build the cohort, freeze the target, split

Features come only from the index session; the target comes only from a later session. The split
is patient-level and stratified, and its hash is recorded so two runs can be shown to have used
identical partitions.


In [ ]:
from neurolens.cohort import build_cohort

cohort = build_cohort(tables, config)
print('analysed:', cohort.audit['n_analysed'], 'of', cohort.audit['n_indexed'])
print('exclusions:', cohort.audit['exclusion_reasons'])
print('target units:', cohort.audit['target_definition']['units'])
print('stage bands:', cohort.audit['target_definition']['stage_bands'])
print('stage distribution:', cohort.audit['stage_distribution'])
print('split:', cohort.split_info['counts'],
      'leakage_free =', cohort.split_info['leakage_free'])
print('split hash:', cohort.split_info['split_hash'])
print('subgroup sizes:', cohort.audit['subgroup_sizes'])


## 5. Run the full pipeline

One call trains every family × comparison × mitigation cell on the untouched training split,
calibrates on validation, evaluates once on test, and writes the audit package.


In [ ]:
from neurolens.pipeline import NeuroLensPipeline

artifacts = NeuroLensPipeline(config).run()
report = artifacts.report
print('run:', artifacts.run_id)
print('artifacts:', len(artifacts.paths))


## 6. What was predicted



In [ ]:
continuous = report['tasks']['continuous']['metrics']
stage = report['tasks']['stage']

print(f"continuous MAE  {continuous['mae']:.4f}  "
      f"95% CI [{continuous['mae_ci'][0]:.4f}, {continuous['mae_ci'][1]:.4f}]  n={continuous['n']}")
print(f"stage macro-F1  {stage['metrics']['macro_f1']:.4f}  "
      f"accuracy {stage['metrics']['accuracy']:.4f}  "
      f"balanced accuracy {stage['metrics']['balanced_accuracy']:.4f}")
print('stage outcome distribution:', stage['outcome_distribution'])
for note in stage['notes']:
    print('  note:', note)


## 7. How certain

Split conformal intervals on the validation residuals give a finite-sample coverage guarantee
under exchangeability. Stated uncertainty is then tested against realised error — if they do not
track each other, abstention based on interval width is **not** supported, and the report says so.


In [ ]:
intervals = report['tasks']['continuous']['intervals']
trust = report['trust']

print(f"PICP {intervals['picp']:.3f} against a nominal "
      f"{1 - report['tasks']['continuous']['alpha']:.2f}; "
      f"mean width {intervals['mean_interval_width']:.3f}; "
      f"Winkler {intervals['winkler_score']:.3f}")
print('calibration:', report['tasks']['continuous']['calibration_regression'])
print('uncertainty vs error:', trust['uncertainty_error']['verdict'],
      f"Spearman {trust['uncertainty_error']['spearman']:.3f}")
print('OOD:', trust['ood']['verdict'])
print('confident and wrong:', trust['confident_and_wrong']['n'], 'of',
      trust['confident_and_wrong']['n_total'])


## 8. Where it fails

Every subgroup row carries **n**, an estimate with a confidence interval, the gap to the reference
group with its own CI, an adequacy status, and a PASS / WARN / INSUFFICIENT DATA verdict. There
is no composite fairness score and no binary fair/unfair label: a subgroup that is too small gets
`INSUFFICIENT DATA` and **no estimate at all**, because absence of a disparity signal there is
not evidence of fairness.


In [ ]:
fairness = pd.DataFrame(report['fairness']['rows'])
fairness[['axis', 'subgroup', 'n', 'metric', 'estimate', 'gap', 'relative_gap',
          'adequacy', 'status']].round(4)


In [ ]:
print('worst group:', report['fairness']['worst_group'])
print()
print('mitigation comparison (identical, untouched test split for every row):')
print(pd.DataFrame(report['fairness']['mitigation_comparison'])
      [['name', 'task', 'mitigation', 'overall', 'worst_group']].round(4))
print()
print('preferred:', report['fairness']['preferred_model'])
print('rationale:', report['fairness']['preferred_rationale'])


## 9. Why — with the causation caveat attached

Patient-level KernelSHAP attributions sum to the prediction minus the base value (the efficiency
identity is asserted in the test suite). **Attributions describe what the fitted model used.**
They are not evidence that a variable causes progression.


In [ ]:
for case in report['explainability']['cases'][:3]:
    print(f"{case['patient_id']}  ({case['selection_reason']})  "
          f"predicted {case['prediction']:+.3f} vs base {case['base_value']:+.3f}  "
          f"residual {case['residual']:.2e}")
    for item in case['top'][:5]:
        print(f"    {item['rank']}. {item['label']:<34} {item['attribution']:+.4f}  "
              f"[{item['family']}]")
    print()


## 10. Can it be reproduced



In [ ]:
import json

manifest = json.load(open(os.path.join(artifacts.run_dir, 'manifest.json'), encoding='utf-8'))
for key in ('run_id', 'config_hash', 'seed', 'dataset', 'dataset_version', 'dataset_access',
            'split_version', 'split_hash', 'target_definition_version'):
    print(f'{key:<26} {manifest[key]}')
print()
print('packages:', manifest['package_versions'])
print()
print('release ready:', manifest['release_readiness']['release_ready'],
      manifest['release_readiness']['blockers'])
print()
print('reproduce with:', report['reproduce_command'])


## 11. Limitations and known failure modes

These are generated from what actually happened in this run — they are not boilerplate.


In [ ]:
print('LIMITATIONS')
for item in report['limitations']:
    print(' -', item)
print()
print('KNOWN FAILURE MODES')
for item in report['known_failure_modes']:
    print(' -', item)


## 12. Open the evidence workspace

The UI is a static single-page app served by the same package. In a local session:

```bash
neurolens serve --runs-root runs --host 0.0.0.0 --port 8000
```

then open <http://localhost:8000>. On Colab, expose the port with a tunnel of your choice, or
download the run directory and serve it locally.


In [ ]:
# from neurolens.server.app import serve
# serve(run_dir=artifacts.run_dir, runs_root='runs', host='0.0.0.0', port=8000)


## 13. The Huntington's disease pathway

The harmonised HD BIDS collections (TRACK-HD, TRACKOn-HD, PREDICT-HD, IMAGE-HD, HD-YAS,
SHIELD-HD, PEARL-HD, LONGPDE10 — 2,216 participants, 7,073 sessions, with FreeSurfer and
fMRIPrep derivatives) are distributed by the CHDI Foundation under a data use agreement obtained
through [Enroll-HD](https://www.enroll-hd.org/for-researchers/datasets/). They are **not** open
data, and NeuroLens never downloads, caches, mirrors, or redistributes them.

If you hold an approved DUA, point the HD configuration at your local copy. Everything
downstream of ingestion is identical to the cells above:

```python
config = NeuroLensConfig.from_yaml('configs/hd-chdi-bids.yaml')
artifacts = NeuroLensPipeline(config).run()
```

That configuration uses UHDRS Total Motor Score as the continuous target, Total Functional
Capacity for the ordinal stage with predeclared cutpoints, and CAG repeat length as the primary
fairness axis and shift axis.


---

**NeuroLens is a research prototype for neurodegenerative disease progression modelling.** It is
not clinically validated, must not be used for diagnosis or treatment decisions, and its feature
attributions describe model association rather than biological causation.
